In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
import random
import os
import time

In [6]:
INPUT = 8
HIDDEN = 32
OUTPUT = 1
EPOCHS = 100

In [3]:
class CutNet(nn.Module):
	def __init__(self):
		super(CutNet, self).__init__()
		self.fc1 = nn.Linear(INPUT, HIDDEN)
		self.fc2 = nn.Linear(HIDDEN, OUTPUT)

	def forward(self, x):
		x = F.relu(self.fc1(x))
		x = self.fc2(x)
		return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net = CutNet().to(device)
optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()

In [4]:
class CutDataset(Dataset):
	def __init__(self, data_file):
		self.data = []
		with open(data_file, 'r') as f:
			for line in f:
				parts = line.strip().split(',')
				if len(parts) != INPUT + 1:
					continue
				features = list(map(float, parts[:-1]))
				label = int(parts[-1])
				self.data.append((features, label))

	def __len__(self):
		return len(self.data)

	def __getitem__(self, idx):
		features, label = self.data[idx]
		return torch.tensor(features, dtype=torch.float32), torch.tensor(label, dtype=torch.float32)

train_dataset = CutDataset('clipped.txt')
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True, num_workers=6, pin_memory=True)


In [7]:
for epoch in range(EPOCHS):
	net.train()
	running_loss = 0.0
	for inputs, labels in train_loader:
		inputs, labels = inputs.to(device), labels.to(device).view(-1, 1)
		optimizer.zero_grad()
		outputs = net(inputs)
		loss = criterion(outputs, labels)
		loss.backward()
		optimizer.step()
		running_loss += loss.item() * inputs.size(0)
	avg_loss = running_loss / len(train_loader.dataset)
	print(f'Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}')
torch.save(net.state_dict(), 'cutnet.pth')

Epoch [1/100], Loss: 0.3972
Epoch [2/100], Loss: 0.3941
Epoch [3/100], Loss: 0.3957
Epoch [4/100], Loss: 0.3950
Epoch [5/100], Loss: 0.3944
Epoch [6/100], Loss: 0.3956
Epoch [7/100], Loss: 0.3936
Epoch [8/100], Loss: 0.3927
Epoch [9/100], Loss: 0.3932
Epoch [10/100], Loss: 0.3929
Epoch [11/100], Loss: 0.3932
Epoch [12/100], Loss: 0.3907
Epoch [13/100], Loss: 0.3921
Epoch [14/100], Loss: 0.3925
Epoch [15/100], Loss: 0.3907
Epoch [16/100], Loss: 0.3916
Epoch [17/100], Loss: 0.3916
Epoch [18/100], Loss: 0.3904
Epoch [19/100], Loss: 0.3908
Epoch [20/100], Loss: 0.3895
Epoch [21/100], Loss: 0.3911
Epoch [22/100], Loss: 0.3909
Epoch [23/100], Loss: 0.3894
Epoch [24/100], Loss: 0.3878
Epoch [25/100], Loss: 0.3889
Epoch [26/100], Loss: 0.3886
Epoch [27/100], Loss: 0.3879
Epoch [28/100], Loss: 0.3891
Epoch [29/100], Loss: 0.3882
Epoch [30/100], Loss: 0.3872
Epoch [31/100], Loss: 0.3875
Epoch [32/100], Loss: 0.3882
Epoch [33/100], Loss: 0.3877
Epoch [34/100], Loss: 0.3860
Epoch [35/100], Loss: 0

In [9]:
# print the weights in c array format
weights = net.state_dict()
for name, param in weights.items():
	if 'weight' in name:
		param = param.cpu().numpy()
		print(f'static const float {name.replace(".", "_")}[] = {{', end='')
		for i in range(param.shape[0]):
			print('{', end='')
			for j in range(param.shape[1]):
				print(f'{param[i][j]:.6f}, ', end='')
			print('}, ', end='')
		print('};')
	elif 'bias' in name:
		param = param.cpu().numpy()
		print(f'static const float {name.replace(".", "_")}[] = {{', end='')
		for i in range(param.shape[0]):
			print(f'{param[i]:.6f}, ', end='')
		print('};')

static const float fc1_weight[] = {{-1.153087, 0.045976, 0.109746, 9.301449, -0.094397, -2.409660, 0.050303, 0.032936, }, {-3.398085, -0.309788, 0.051887, -5.799692, 0.303075, -1.365884, -0.018369, -0.002675, }, {0.267782, -0.416899, -0.010965, -2.014424, 0.061046, -0.408126, -0.167140, 0.217714, }, {-0.593143, -1.043535, 0.038595, 7.061245, 0.241283, -0.149503, 0.007277, 0.000415, }, {-0.218181, -0.062414, 0.008903, -18.010000, 0.181720, 1.293801, 0.027030, 0.008878, }, {-3.526786, -0.160954, -0.001377, -7.861702, 0.336685, 0.899523, 0.000985, 0.000396, }, {1.250485, 0.243583, 0.102928, 1.176287, -0.193379, 0.403101, 0.084018, -0.002799, }, {2.942957, -0.397860, 0.258021, -10.376847, -0.239111, 1.574958, 0.016172, -0.011779, }, {-1.968534, -1.174010, -0.000643, -0.375598, 0.267682, -1.792346, 0.009275, 0.004078, }, {-1.506415, -0.659181, 0.012934, -4.040979, -0.172446, -1.000159, 0.003880, 0.000819, }, {-2.257093, -2.522721, 0.009397, -10.140518, 0.089560, -1.420032, 0.003487, 0.00053